In [11]:
%pwd

'f:\\Desktop\\DATA SCIENCE\\Topics (Left)\\MLOPs\\CI-CD + Monitoring + Orchestration\\Tutorial'

In [2]:
%cd ..

f:\Desktop\DATA SCIENCE\Topics (Left)\MLOPs\CI-CD + Monitoring + Orchestration\Tutorial


In [12]:
import os
import sys
from dataclasses import dataclass
from pathlib import Path
from mlops.constants import CONFIG_FILE_PATH
from mlops.utils.common import read_yaml, create_dir
from mlops.exception import CustomException
import shutil
from sklearn.model_selection import train_test_split
import pandas as pd

In [10]:
from mlops.logger import setup_logger
logging = setup_logger()

In [13]:
# entity
@dataclass(frozen=True)
class DataIngestionConfig:
    raw_data_path: Path
    ingestion_dir: Path
    raw_data_file: Path
    train_data_file: Path
    test_data_file: Path

In [ ]:
# from box import ConfigBox
# import yaml
# with open("config/config.yaml") as f:
#     y = yaml.safe_load(f)

# cy = ConfigBox(y)

# print(cy)
# print(cy.artifacts_root)
# print(cy.data_ingestion)
# print(cy.data_ingestion.raw_data_path)
# print(cy.data_ingestion.ingestion_dir)

{'artifacts_root': 'artifacts', 'data_ingestion': {'raw_data_path': 'data/raw/stud.csv', 'ingestion_dir': 'artifacts/data_ingestion'}}
artifacts
{'raw_data_path': 'data/raw/stud.csv', 'ingestion_dir': 'artifacts/data_ingestion'}
data/raw/stud.csv
artifacts/data_ingestion


In [14]:
# Configuration
class ConfigurationManager:
    def __init__(self, config_filepath = CONFIG_FILE_PATH):
        self.config = read_yaml(config_filepath)
        create_dir([self.config.artifacts_root])    # artifacts dir is created

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion
        create_dir([config.ingestion_dir])           # artifacts/data_ingestion is created

        data_ingestion_config = DataIngestionConfig(
                                raw_data_path=Path(config.raw_data_path),
                                ingestion_dir=Path(config.ingestion_dir),
                                raw_data_file=Path(config.raw_data_file),
                                train_data_file=Path(config.train_data_file),
                                test_data_file=Path(config.test_data_file)
                                )
        
        return data_ingestion_config
        

In [ ]:
# c = ConfigurationManager()
# con = c.get_data_ingestion_config()
# print(con)
# print(con.ingestion_dir)
# print(con.raw_data_file)

DataIngestionConfig(raw_data_path='data/raw/stud.csv', ingestion_dir='artifacts/data_ingestion', raw_data_file='artifacts/data_ingestion/raw.csv', train_data_file='artifacts/data_ingestion/train.csv', test_data_file='artifacts/data_ingestion/test.csv')
artifacts/data_ingestion
artifacts/data_ingestion/raw.csv


In [15]:
# components/data_ingestion.py
class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config

    def get_dataset(self) -> Path:
        """Fetch dataset from raw folder and copy to artifacts."""
        try:
            source_path = self.config.raw_data_path
            destination_file = self.config.raw_data_file

            logging.info("----- Data ingestion has started ------")

            shutil.copy(source_path, destination_file)

            logging.info(f"Dataset copied from {source_path} to {destination_file}")

            return destination_file
        
        except Exception as e:
            raise CustomException(e, sys)

    def train_test_split(self) -> tuple[Path, Path]:
        """Create train and test split"""

        try:
            logging.info("Creation of train & test set from raw data started --->")

            df = pd.read_csv(self.config.raw_data_path)
            train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

            train_df.to_csv(self.config.train_data_file, index=False)
            test_df.to_csv(self.config.test_data_file, index=False)

            logging.info("Train & test set successfully created")

        except Exception as e:
            raise CustomException(e,sys)


In [16]:
# pipeline/01_data_ingestion.py
try:
    logging.info("Data ingestion Pipeline started ----->")
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.get_dataset()
    data_ingestion.train_test_split()
    logging.info("Data ingestion Pipeline executed successfully.")

except Exception as e:
    raise CustomException(e,sys)